In [ ]:
# import numpy as np
# import json
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# from matplotlib.pyplot import hist
# from scipy.stats import pearsonr
#
# # ==========================================
# # 1. LOAD THE DATA
# # ==========================================
# # Update this path if your files are in a specific folder
# data_dir = "./dev_perturb_adamson-Jun28-15-05"
#
# print("Loading data...")
# results = np.load(f"{data_dir}/test_results.npz", allow_pickle=True)
# pred = results['pred']
# truth = results['truth']
# pert_cat = results['pert_cat']
#
# with open(f"{data_dir}/gene_names.json", "r") as f:
#     gene_names = np.array(json.load(f))
#
# print(f"Loaded {len(pert_cat)} cells and {len(gene_names)} genes.")
#
# # ==========================================
# # 2. ALIGN GENE NAMES WITH DATA
# # ==========================================
# # STEP A: Fix the 2322 vs 2320 mismatch.
# # The model internally dropped the hyphenated dummy genes we injected.
# mismatch_mask = ~np.isin(gene_names, ["IR-24h", "IR-1week"])
# gene_names = gene_names[mismatch_mask]
#
# # Quick check to ensure they match perfectly now
# assert truth.shape[1] == len(gene_names), f"Mismatch! Data: {truth.shape[1]}, Names: {len(gene_names)}"
#
# # STEP B: Remove the old underscored dummy genes from the analysis entirely
# # We must remove them from BOTH the names list AND the data arrays simultaneously
# plot_mask = ~np.isin(gene_names, ["IR_24h", "IR_1week"])
#
# gene_names = gene_names[plot_mask]
# truth = truth[:, plot_mask]
# pred = pred[:, plot_mask]
#
# print(f"Final aligned shape -> Data: {truth.shape[1]} genes, Names: {len(gene_names)} genes")

In [ ]:
# # ==========================================
# # 3. CALCULATE GENE CHANGES (DELTAS) for 24h
# # ==========================================
# target_pert = "IR-24h"  # Change to "IR-1week" if you want to check the other condition
#
# ctrl_mask = (pert_cat == 'ctrl')
# pert_mask = (pert_cat == target_pert)
#
# if ctrl_mask.sum() == 0 or pert_mask.sum() == 0:
#     raise ValueError(f"Missing control or {target_pert} cells in the test set!")
#
# # Mean expression per gene across all cells in that condition
# mean_ctrl_true = truth[ctrl_mask].mean(axis=0)
# mean_pert_true = truth[pert_mask].mean(axis=0)
# mean_pert_pred = pred[pert_mask].mean(axis=0)
#
# # True Delta (What actually happened biologically)
# delta_true = mean_pert_true - mean_ctrl_true
# # Predicted Delta (What the model guessed would happen)
# delta_pred = mean_pert_pred - mean_ctrl_true
#
# # ==========================================
# # 4. FIND TOP DIFFERENTIALLY EXPRESSED GENES
# # ==========================================
# num_genes_to_select = 10
# sorted_indices = np.argsort(delta_true)
# down_reg_idx = sorted_indices[:num_genes_to_select]     # The left tail (Negative Delta)
# up_reg_idx = sorted_indices[-num_genes_to_select:]      # The right tail (Positive Delta)
#
# print("\n--- Selected Phenotype Genes ---")
# print(f"Down-regulated: {', '.join(gene_names[down_reg_idx])}")
# print(f"Up-regulated: {', '.join(gene_names[up_reg_idx])}")
#
# # ==========================================
# # 6. PLOT 2: THE SELECTED PHENOTYPES BAR CHART
# # ==========================================
# # Combine the selected tails
# selected_indices = np.concatenate([down_reg_idx, up_reg_idx])
# selected_genes = gene_names[selected_indices]
# sel_true_vals = delta_true[selected_indices]
# sel_pred_vals = delta_pred[selected_indices]
#
# # Create DataFrame for clean plotting
# df_plot = pd.DataFrame({
#     'Gene': selected_genes,
#     'True $\Delta$': sel_true_vals,
#     'Predicted $\Delta$': sel_pred_vals
# })
# df_plot.set_index('Gene', inplace=True)
# df_plot = df_plot.sort_values(by='True $\Delta$') # Sort from most negative to most positive
#
# # Plotting the Bar Chart
# fig, ax = plt.subplots(figsize=(14, 6))
# df_plot.plot(kind='bar', ax=ax, color=['#4C72B0', '#DD8452'], edgecolor='black', width=0.8)
#
# plt.title(f"Model Accuracy on Selected Phenotypes: Extremes of the Histogram ({target_pert})", fontsize=14)
# plt.ylabel("Change in Expression over Control ($\Delta$)", fontsize=12)
# plt.xlabel("Phenotype Genes", fontsize=12)
# plt.axhline(0, color='black', linewidth=1)
#
# plt.xticks(rotation=45, ha='right', fontsize=11)
# plt.grid(axis='y', linestyle='--', alpha=0.7)
# plt.legend(fontsize=12)
# plt.tight_layout()
#
# plt.savefig(f"Phenotype_BarChart_{target_pert}.png", dpi=300)
# plt.show()

In [ ]:
# # ==========================================
# # 4. SELECT THE TOP 20 GENES
# # ==========================================
# # Find 10 most down-regulated and 10 most up-regulated genes
# mean_true_deltas = sel_true_vals.mean(axis=0)
# sorted_indices = np.argsort(mean_true_deltas)
#
# top_down_idx = sorted_indices[:10]
# top_up_idx = sorted_indices[-10:]
# selected_idx = np.concatenate([top_down_idx, top_up_idx])
#
# selected_genes = gene_names[selected_idx]
#
# # Extract the data arrays for these specific 20 genes
# boxplot_data = [sel_true_vals[:, i] for i in selected_idx]
# red_dots_data = [sel_pred_vals.mean(axis=0)[i] for i in selected_idx]
#
# # ==========================================
# # 5. CREATE THE scGPT-STYLE PLOT
# # ==========================================
# fig, ax = plt.subplots(figsize=(14, 5.5))
#
# # Draw the Boxplots (True cellular distributions)
# bp = ax.boxplot(boxplot_data, showfliers=False, patch_artist=True, widths=0.5)
#
# # Style them to match the tutorial (White boxes, black lines)
# for box in bp['boxes']:
#     box.set_facecolor('white')
#     box.set_edgecolor('black')
#     box.set_linewidth(1)
# for whisker in bp['whiskers']:
#     whisker.set_color('black')
# for cap in bp['caps']:
#     cap.set_color('black')
# for median in bp['medians']:
#     median.set_color('gray')  # Make median less distracting so the red dot pops
#
# # Draw the Red Dots (Model's prediction)
# x_coords = np.arange(1, len(selected_genes) + 1)
# ax.scatter(x_coords, red_dots_data, color='red', s=30, zorder=5, label='Predicted Mean $\Delta$')
#
# # Draw the Green baseline (Zero change)
# ax.axhline(0, color='forestgreen', linestyle='dashed', linewidth=1.2)
#
# # Formatting
# ax.set_xticks(x_coords)
# ax.set_xticklabels(selected_genes, rotation=90, fontsize=11)
# ax.set_ylabel("Change in Gene Expression over Control", fontsize=13, labelpad=10)
# ax.set_title(f"{target_pert} + ctrl", fontsize=15, pad=15)
#
# # Clean up borders like the tutorial
# sns.despine(trim=False)
# ax.spines['bottom'].set_linewidth(1.5)
# ax.spines['left'].set_linewidth(1.5)
#
# plt.tight_layout()
# # plt.savefig(f"scGPT_Boxplot_{target_pert}.png", dpi=300)
# plt.show()
# train_data = pd.read_csv(f'{data_dir}/train_metadata.csv')
# plt.hist(train_data['Timepoint'])
# plt.show()
# print(train_data.shape)

# Formatted AnnData Dataset Informatics

This section describes the full formatted dataset used for scGPT perturbation prediction. It reads `data/adamson/perturb_processed.h5ad` directly rather than using train, validation, or test metadata exported from a single model run.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

PROJECT_DIR = Path("/Users/zhehaowang/PycharmProjects/scgpt_test")
H5AD_PATH = PROJECT_DIR / "data" / "adamson" / "perturb_processed.h5ad"
OUTPUT_DIR = PROJECT_DIR / "analysis_outputs" / "formatted_dataset_informatics"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONDITION_ORDER = ["ctrl", "IR-24h", "IR-1week"]
SAMPLE_PANEL_CONDITION_ORDER = ["IR-1week", "IR-24h", "ctrl"]
BROAD_SUBTYPE_ORDER = ["AC-like", "MES-like", "NPC-like", "OPC-like"]
DETAILED_SUBTYPE_ORDER = [
    "AC-like",
    "AC-like Prolif",
    "MES-like hypoxia independent",
    "MES-like hypoxia/MHC",
    "NPC-like OPC",
    "NPC-like Prolif",
    "NPC-like neural",
    "OPC-like",
    "OPC-like Prolif",
]
QC_COLUMNS = ["n_genes", "total_counts", "pct_counts_mito", "pct_counts_ribo"]

In [ ]:
def _decode(value):
    return value.decode() if isinstance(value, bytes) else value


def read_h5ad_obs_with_h5py(path):
    import h5py

    with h5py.File(path, "r") as handle:
        obs_group = handle["obs"]
        index_name = obs_group.attrs["_index"]
        index = [_decode(value) for value in obs_group[index_name][:]]
        columns = [_decode(value) for value in obs_group.attrs["column-order"]]
        data = {}
        for column in columns:
            obj = obs_group[column]
            if hasattr(obj, "attrs") and obj.attrs.get("encoding-type") == "categorical":
                categories = [_decode(value) for value in obj["categories"][:]]
                codes = obj["codes"][:]
                data[column] = [pd.NA if code < 0 else categories[int(code)] for code in codes]
            else:
                data[column] = [_decode(value) for value in obj[:]]
        n_vars = int(handle["X"].attrs["shape"][1])
    return pd.DataFrame(data, index=index), n_vars


try:
    import scanpy as sc
    adata = sc.read_h5ad(H5AD_PATH)
    metadata = adata.obs.copy()
    n_vars = int(adata.n_vars)
    read_source = "scanpy"
except ModuleNotFoundError:
    metadata, n_vars = read_h5ad_obs_with_h5py(H5AD_PATH)
    read_source = "h5py fallback"

metadata["condition"] = metadata["condition"].replace({"IR_24h": "IR-24h", "IR_1week": "IR-1week"})
if "condition_name" in metadata:
    metadata["condition_name"] = metadata["condition_name"].replace({"IR_24h": "IR-24h", "IR_1week": "IR-1week"})
for column in QC_COLUMNS:
    metadata[column] = pd.to_numeric(metadata[column], errors="coerce")

print(f"Read source: {read_source}")
print(f"Cells: {metadata.shape[0]:,}")
print(f"Genes: {n_vars:,}")
display(metadata.head())

In [ ]:
dataset_summary = pd.DataFrame([
    {"metric": "cells", "value": metadata.shape[0]},
    {"metric": "genes", "value": n_vars},
    {"metric": "samples", "value": metadata["Sample"].nunique()},
    {"metric": "batches", "value": metadata["Batch"].nunique()},
    {"metric": "broad_malignant_states", "value": metadata["GBM_malig_subtypes"].nunique()},
    {"metric": "detailed_malignant_states", "value": metadata["GBM_malig_detailed"].nunique()},
    {"metric": "read_source", "value": read_source},
])

condition_counts = (
    metadata["condition"]
    .value_counts()
    .reindex(CONDITION_ORDER)
    .rename_axis("condition")
    .reset_index(name="n_cells")
)
condition_counts["percent_total"] = 100 * condition_counts["n_cells"] / len(metadata)

sample_summary = (
    metadata.groupby(["condition", "Sample", "Batch", "Sex"], dropna=False, observed=True)
    .size()
    .reset_index(name="n_cells")
    .sort_values(["condition", "Sample"])
)
condition_sample_counts = pd.crosstab(metadata["condition"], metadata["Sample"]).reindex(CONDITION_ORDER).fillna(0).astype(int)

display(dataset_summary)
display(condition_counts)
display(sample_summary)
display(condition_sample_counts)

In [ ]:
condition_broad_counts = pd.crosstab(metadata["condition"], metadata["GBM_malig_subtypes"]).reindex(CONDITION_ORDER)[BROAD_SUBTYPE_ORDER]
condition_broad_percent = condition_broad_counts.div(condition_broad_counts.sum(axis=1), axis=0) * 100

sample_broad_counts = (
    pd.crosstab([metadata["condition"], metadata["Sample"]], metadata["GBM_malig_subtypes"])
    .reindex(columns=BROAD_SUBTYPE_ORDER)
    .fillna(0)
    .astype(int)
    .sort_index()
)
sample_broad_percent = sample_broad_counts.div(sample_broad_counts.sum(axis=1), axis=0) * 100

condition_detailed_counts = pd.crosstab(metadata["condition"], metadata["GBM_malig_detailed"]).reindex(CONDITION_ORDER)[DETAILED_SUBTYPE_ORDER]
condition_detailed_percent = condition_detailed_counts.div(condition_detailed_counts.sum(axis=1), axis=0) * 100

survival_condition_counts = pd.crosstab(
    metadata["condition"],
    metadata["survival_classification"].fillna("NA"),
).reindex(CONDITION_ORDER)[["NA", "dying", "surviving"]]

display(condition_broad_counts)
display(condition_broad_percent.round(1))
display(sample_broad_counts)
display(sample_broad_percent.round(1))
display(condition_detailed_counts)
display(condition_detailed_percent.round(1))
display(survival_condition_counts)

In [ ]:
def median_iqr_table(data, group_col, value_cols):
    rows = []
    for group, group_df in data.groupby(group_col, sort=False):
        row = {group_col: group, "n_cells": len(group_df)}
        for col in value_cols:
            q1, median, q3 = group_df[col].quantile([0.25, 0.5, 0.75])
            row[f"{col}_median"] = median
            row[f"{col}_q1"] = q1
            row[f"{col}_q3"] = q3
            row[f"{col}_median_iqr"] = f"{median:.2f} [{q1:.2f}, {q3:.2f}]"
        rows.append(row)
    result = pd.DataFrame(rows)
    result[group_col] = pd.Categorical(result[group_col], CONDITION_ORDER, ordered=True)
    result = result.sort_values(group_col)
    result[group_col] = result[group_col].astype(str)
    return result


qc_by_condition = median_iqr_table(metadata, "condition", QC_COLUMNS)
qc_display_cols = ["condition", "n_cells"] + [f"{col}_median_iqr" for col in QC_COLUMNS]
display(qc_by_condition[qc_display_cols])

In [ ]:
summary_tables = {
    "dataset_summary": dataset_summary,
    "condition_counts": condition_counts,
    "sample_summary": sample_summary,
    "condition_sample_counts": condition_sample_counts,
    "condition_broad_counts": condition_broad_counts,
    "condition_broad_percent": condition_broad_percent,
    "sample_broad_counts": sample_broad_counts,
    "sample_broad_percent": sample_broad_percent,
    "condition_detailed_counts": condition_detailed_counts,
    "condition_detailed_percent": condition_detailed_percent,
    "survival_condition_counts": survival_condition_counts,
    "qc_by_condition": qc_by_condition,
}

for name, table in summary_tables.items():
    table.to_csv(OUTPUT_DIR / f"{name}.csv")

print(f"Saved {len(summary_tables)} tables to {OUTPUT_DIR}")

## Visualisations

The figures below are generated from the same `perturb_processed.h5ad` source and saved as PNG files for manuscript drafting. The subtype labels are wrapped to keep them clear of the heatmap cells.

In [ ]:
try:
    import textwrap

    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    condition_palette = {"ctrl": "#355C7D", "IR-24h": "#E76F51", "IR-1week": "#2A9D8F"}
    subtype_palette = {"AC-like": "#457B9D", "MES-like": "#D1495B", "NPC-like": "#2A9D8F", "OPC-like": "#8E6C8A"}
    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.titleweight": "bold",
        "figure.dpi": 130,
        "savefig.dpi": 300,
    })

    def wrap_label(label, width=13):
        return "\n".join(textwrap.wrap(str(label), width=width, break_long_words=False))

    def save_figure(fig, path):
        fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")

    def style_bar_axis(ax):
        ax.grid(axis="y", color="#E6E6E6", linewidth=0.8)
        ax.set_axisbelow(True)

    def sample_panel_dataframe(sample_summary):
        sample_counts = (
            sample_summary.groupby(["condition", "Sample"], as_index=False, observed=True)["n_cells"]
            .sum()
            .query("n_cells > 0")
        )
        rows = []
        for condition in SAMPLE_PANEL_CONDITION_ORDER:
            condition_rows = sample_counts[sample_counts["condition"] == condition].sort_values("Sample").copy()
            condition_rows["display_sample"] = [f"{condition}_{i}" for i in range(1, len(condition_rows) + 1)]
            rows.append(condition_rows)
        return pd.concat(rows, ignore_index=True)

    fig, axes = plt.subplots(2, 1, figsize=(9.2, 8.0), gridspec_kw={"height_ratios": [1.05, 1]})
    fig.suptitle("Formatted AnnData dataset composition", x=0.01, y=0.995, ha="left", fontsize=15, fontweight="bold")
    condition_plot = condition_counts.set_index("condition").loc[CONDITION_ORDER]
    condition_ymax = condition_plot["n_cells"].max() * 1.28
    axes[0].bar(CONDITION_ORDER, condition_plot["n_cells"], color=[condition_palette[c] for c in CONDITION_ORDER], width=0.62)
    axes[0].set_title("A  Radiotherapy condition balance", loc="left", pad=16)
    axes[0].set_ylabel("Cells")
    axes[0].set_ylim(0, condition_ymax)
    style_bar_axis(axes[0])
    for i, (condition, row) in enumerate(condition_plot.iterrows()):
        axes[0].text(
            i,
            row["n_cells"] + condition_ymax * 0.035,
            f"{int(row['n_cells']):,}\n{row['percent_total']:.1f}%",
            ha="center",
            va="bottom",
            fontweight="bold",
            linespacing=0.95,
        )

    sample_plot = sample_panel_dataframe(sample_summary)
    axes[1].bar(sample_plot["display_sample"], sample_plot["n_cells"], color=[condition_palette[c] for c in sample_plot["condition"]], width=0.72)
    axes[1].set_title("B  Sample-level cell counts", loc="left", pad=16)
    axes[1].set_ylabel("Cells")
    axes[1].tick_params(axis="x", rotation=45)
    style_bar_axis(axes[1])
    axes[1].legend(handles=[Patch(facecolor=condition_palette[c], label=c) for c in SAMPLE_PANEL_CONDITION_ORDER], loc="upper center", bbox_to_anchor=(0.5, -0.28), ncol=3, frameon=False)
    fig.subplots_adjust(top=0.90, bottom=0.17, hspace=0.55)
    save_figure(fig, OUTPUT_DIR / "figure_h5ad_condition_sample_composition.png")
    save_figure(fig, OUTPUT_DIR / "h5ad_condition_sample_composition.png")
    fig.savefig(OUTPUT_DIR / "figure_h5ad_condition_sample_composition.pdf", bbox_inches="tight", facecolor="white")
    fig.savefig(OUTPUT_DIR / "h5ad_condition_sample_composition.pdf", bbox_inches="tight", facecolor="white")
    display(fig)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(11.5, 4.9))
    detailed_percent = condition_detailed_percent.loc[CONDITION_ORDER, DETAILED_SUBTYPE_ORDER]
    detailed_counts = condition_detailed_counts.loc[CONDITION_ORDER, DETAILED_SUBTYPE_ORDER]
    im = ax.imshow(detailed_percent, cmap="YlGnBu", aspect="auto", vmin=0, vmax=40)
    ax.set_title("Detailed malignant-cell subtype composition", loc="left", pad=18)
    ax.set_yticks(range(len(CONDITION_ORDER)))
    ax.set_yticklabels(CONDITION_ORDER, fontweight="bold")
    ax.set_xticks(range(len(DETAILED_SUBTYPE_ORDER)))
    ax.set_xticklabels([wrap_label(label) for label in DETAILED_SUBTYPE_ORDER], rotation=0, ha="center")
    ax.tick_params(axis="x", length=0, pad=10, labelsize=8)
    ax.tick_params(axis="y", length=0, pad=8)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xticks([x - 0.5 for x in range(1, len(DETAILED_SUBTYPE_ORDER))], minor=True)
    ax.set_yticks([y - 0.5 for y in range(1, len(CONDITION_ORDER))], minor=True)
    ax.grid(which="minor", color="white", linewidth=4)
    ax.tick_params(which="minor", bottom=False, left=False)
    for i, condition in enumerate(CONDITION_ORDER):
        for j, subtype in enumerate(DETAILED_SUBTYPE_ORDER):
            pct = detailed_percent.loc[condition, subtype]
            count = detailed_counts.loc[condition, subtype]
            ax.text(
                j,
                i,
                f"{pct:.1f}%\n{int(count):,}",
                ha="center",
                va="center",
                fontsize=8,
                fontweight="bold" if pct >= 15 else "normal",
                color="white" if pct >= 20 else "#222222",
                linespacing=1.35,
            )
    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label("Cell percentage within condition")
    cbar.outline.set_visible(False)
    fig.subplots_adjust(top=0.80, bottom=0.28, left=0.08, right=0.94)
    save_figure(fig, OUTPUT_DIR / "figure_h5ad_detailed_subtype_composition.png")
    save_figure(fig, OUTPUT_DIR / "h5ad_detailed_subtype_composition.png")
    fig.savefig(OUTPUT_DIR / "figure_h5ad_detailed_subtype_composition.pdf", bbox_inches="tight", facecolor="white")
    fig.savefig(OUTPUT_DIR / "h5ad_detailed_subtype_composition.pdf", bbox_inches="tight", facecolor="white")
    display(fig)
    plt.close(fig)

    qc_specs = [
        ("n_genes", "Detected genes per cell", "Number of detected genes"),
        ("total_counts", "Total counts per cell", "Total expression counts"),
        ("pct_counts_mito", "Mitochondrial counts", "Mitochondrial counts (%)"),
        ("pct_counts_ribo", "Ribosomal counts", "Ribosomal counts (%)"),
    ]
    qc_table = qc_by_condition.set_index("condition")
    fig, axes = plt.subplots(2, 2, figsize=(10.4, 6.6))
    fig.suptitle("Quality-control summaries by radiotherapy condition", x=0.01, y=0.995, ha="left", fontsize=15, fontweight="bold")
    for ax, (metric, title, x_label) in zip(axes.flat, qc_specs):
        values = []
        for condition in CONDITION_ORDER:
            row = qc_table.loc[condition]
            values.extend([float(row[f"{metric}_q1"]), float(row[f"{metric}_median"]), float(row[f"{metric}_q3"])])
        xmin, xmax = min(values), max(values)
        pad = (xmax - xmin) * 0.15 if xmax > xmin else 1
        xmin = max(0, xmin - pad)
        xmax += pad
        ax.set_title(title, loc="left", pad=10)
        for y, condition in enumerate(CONDITION_ORDER):
            row = qc_table.loc[condition]
            q1 = float(row[f"{metric}_q1"])
            median = float(row[f"{metric}_median"])
            q3 = float(row[f"{metric}_q3"])
            ax.hlines(y, q1, q3, color=condition_palette[condition], linewidth=6)
            ax.plot(median, y, "o", color=condition_palette[condition], markeredgecolor="white", markersize=8)
            ax.text(q3 + (xmax - xmin) * 0.015, y, f"{median:.2f}", va="center", fontsize=8, color="#555555")
        ax.set_yticks(range(len(CONDITION_ORDER)))
        ax.set_yticklabels(CONDITION_ORDER)
        ax.invert_yaxis()
        ax.set_xlim(xmin, xmax)
        ax.set_xlabel(x_label)
        ax.grid(axis="x", color="#E6E6E6", linewidth=0.8)
        ax.spines["left"].set_visible(False)
        ax.tick_params(axis="y", length=0)
    fig.subplots_adjust(top=0.87, bottom=0.10, left=0.10, right=0.98, hspace=0.55, wspace=0.32)
    save_figure(fig, OUTPUT_DIR / "figure_h5ad_qc_summary.png")
    fig.savefig(OUTPUT_DIR / "figure_h5ad_qc_summary.pdf", bbox_inches="tight", facecolor="white")
    display(fig)
    plt.close(fig)
except ModuleNotFoundError as exc:
    print(f"Matplotlib plotting skipped: {exc}")

In [ ]:
from IPython.display import Image, display

figure_paths = [
    OUTPUT_DIR / "figure_h5ad_condition_sample_composition.png",
    OUTPUT_DIR / "figure_h5ad_subtype_sample_composition.png",
    OUTPUT_DIR / "figure_h5ad_detailed_subtype_composition.png",
    OUTPUT_DIR / "figure_h5ad_qc_summary.png",
]

for figure_path in figure_paths:
    if figure_path.exists():
        print(figure_path)
        display(Image(filename=str(figure_path)))
    else:
        print(f"Missing figure: {figure_path}")

**Sample-level state-composition figure caption.** (A) Pooled proportions of AC-, MES-, NPC- and OPC-like malignant cells in control, IR-24h and IR-1week conditions. (B) Corresponding within-sample proportions shown as 100% stacked bars for each of the four samples per condition. Sample labels are condition-specific display labels ordered within condition. The replicate-level view tests whether pooled state shifts are shared across samples rather than being driven by unequal cell recovery in one animal.

## Draft Result Text

The formatted AnnData object contained 19,037 malignant cells and 2,467 features after preprocessing. Cells were distributed across three radiotherapy conditions: control (12,634 cells, 66.4%), IR-24h (4,536 cells, 23.8%), and IR-1week (1,867 cells, 9.8%). The dataset contained 12 samples across four batches. Malignant cells were annotated into four broad transcriptional states and nine detailed malignant-cell subtypes.

Pooled composition shifted from an NPC-like-dominant control population (62.6% NPC-like and 13.6% MES-like) towards a more mesenchymal state after radiotherapy. At 24 hours, the MES-like fraction increased to 25.2% while the NPC-like fraction decreased to 53.7%; at one week these fractions were 35.1% and 37.1%, respectively. The sample-level analysis showed the same direction in every batch-matched replicate. MES-like cells represented 29.9-39.1% of individual one-week samples compared with 9.1-21.5% of control samples, whereas NPC-like cells represented 34.5-41.7% at one week compared with 54.6-67.8% in controls. These consistent within-sample shifts indicate that the pooled MES-like increase and NPC-like decrease were not explained by one disproportionately recovered sample.

# scGPT Model Evaluation

This section evaluates the saved scGPT run `dev_perturb_adamson-Jun28-15-05` using the train, validation and test split files exported by that run. The section also checks the saved prediction arrays before using them for pseudobulk diagnostics.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)

PROJECT_DIR = Path("/Users/zhehaowang/PycharmProjects/scgpt_test")
RUN_DIR = PROJECT_DIR / "dev_perturb_adamson-Jun28-15-05"
OUTPUT_DIR = PROJECT_DIR / "analysis_outputs" / "scgpt_model_evaluation"

print(f"Run directory: {RUN_DIR}")
print(f"Evaluation outputs: {OUTPUT_DIR}")

In [ ]:
split_summary = pd.read_csv(OUTPUT_DIR / "split_summary.csv", index_col=0)
split_condition_counts = pd.read_csv(OUTPUT_DIR / "split_condition_counts.csv", index_col=0)
split_condition_percent = pd.read_csv(OUTPUT_DIR / "split_condition_percent.csv", index_col=0)
result_integrity_checks = pd.read_csv(OUTPUT_DIR / "result_integrity_checks.csv", index_col=0)
result_condition_blocks = pd.read_csv(OUTPUT_DIR / "result_condition_blocks.csv", index_col=0)

display(split_summary)
display(split_condition_counts)
display(split_condition_percent.round(1))
display(result_integrity_checks)
display(result_condition_blocks)

**Prediction-array scope and leakage note.** The saved `test_results.npz` file contains 19,037 rows rather than the 1,904 nominal test cells. Exact hashing of each truth-expression row against the saved train, validation and test matrices recovered a unique metadata match for all 19,037 rows. This makes state-, sample- and split-aware diagnostics possible, and the single-cell figures below use only rows mapped to the nominal test split. However, the run constructed GEARS loaders from condition membership while all three conditions were present in every nominal split; consequently the saved evaluation arrays cover the full dataset and this run does not provide a clean held-out generalisation estimate. Its results should be reported as descriptive reconstruction performance until the model is rerun with genuinely disjoint cell-level loaders.

**What the gene-only pseudobulk delta diagnostic means.** The AnnData feature axis contains gene-expression variables plus condition-indicator variables, so the diagnostic first removes the non-gene condition indicators. The remaining expression values are averaged within each condition to form pseudobulk profiles. Delta expression is then calculated relative to the control mean. Plotting $\Delta_{\mathrm{observed}}$ against $\Delta_{\mathrm{predicted}}$ asks whether scGPT recovers the direction and magnitude of radiation-induced gene-expression changes, rather than only matching absolute expression levels dominated by genes that do not change.

In [ ]:
validation_simple_metrics = pd.read_csv(OUTPUT_DIR / "validation_simple_metrics.csv", index_col=0)
validation_full_metrics = pd.read_csv(OUTPUT_DIR / "validation_full_metrics.csv", index_col=0)
validation_report_metrics = pd.read_csv(OUTPUT_DIR / "validation_report_metrics.csv", index_col=0)
updated_validation_local_auprc_metrics = pd.read_csv(
    OUTPUT_DIR / "updated_validation_local_auprc_metrics.csv", index_col=0
)
updated_test_local_auprc_metrics = pd.read_csv(
    OUTPUT_DIR / "updated_test_local_auprc_metrics.csv", index_col=0
)
validation_checkpoint_auprc_metrics = pd.read_csv(
    OUTPUT_DIR / "validation_checkpoint_auprc_metrics.csv", index_col=0
)
validation_normalised_auprc_metrics = pd.read_csv(
    OUTPUT_DIR / "validation_normalised_auprc_metrics.csv", index_col=0
)
test_final_metrics = pd.read_csv(OUTPUT_DIR / "test_final_metrics.csv", index_col=0)
test_report_metrics = pd.read_csv(OUTPUT_DIR / "test_report_metrics.csv", index_col=0)
best_validation_epoch = pd.read_csv(OUTPUT_DIR / "best_validation_epoch_by_average_delta.csv", index_col=0)
pseudobulk_gene_only_metrics = pd.read_csv(OUTPUT_DIR / "pseudobulk_gene_only_metrics.csv", index_col=0)
pseudobulk_expression_level_metrics = pd.read_csv(OUTPUT_DIR / "pseudobulk_expression_level_metrics.csv", index_col=0)
delta_linearity_diagnostics = pd.read_csv(OUTPUT_DIR / "delta_linearity_diagnostics.csv", index_col=0)
top20_de_genes = pd.read_csv(OUTPUT_DIR / "top20_differentially_expressed_genes.csv", index_col=0)
result_metadata_alignment_summary = pd.read_csv(OUTPUT_DIR / "result_metadata_alignment_summary.csv", index_col=0)
heterogeneity_variance_retention = pd.read_csv(OUTPUT_DIR / "heterogeneity_variance_retention.csv", index_col=0)
state_delta_metrics = pd.read_csv(OUTPUT_DIR / "state_delta_metrics.csv", index_col=0)
state_centroid_separation = pd.read_csv(OUTPUT_DIR / "state_centroid_separation.csv", index_col=0)
pathway_score_changes = pd.read_csv(OUTPUT_DIR / "pathway_score_changes.csv", index_col=0)
pathway_score_recovery_metrics = pd.read_csv(OUTPUT_DIR / "pathway_score_recovery_metrics.csv", index_col=0)
pathway_gene_sets = pd.read_csv(OUTPUT_DIR / "pathway_gene_sets.csv", index_col=0)

display(best_validation_epoch)
display(updated_validation_local_auprc_metrics.tail())
display(updated_test_local_auprc_metrics)
display(validation_checkpoint_auprc_metrics)
display(validation_normalised_auprc_metrics.tail())
display(test_report_metrics)
display(result_metadata_alignment_summary)
display(pseudobulk_expression_level_metrics)
display(pseudobulk_gene_only_metrics)
display(delta_linearity_diagnostics)
display(heterogeneity_variance_retention)
display(state_delta_metrics)
display(state_centroid_separation)
display(pathway_score_changes)
display(pathway_score_recovery_metrics)
display(pathway_gene_sets)

## Main Model and Single-Cell Results

The heterogeneity-recovery composite is the central single-cell result. Panel A quantifies the fraction of observed within-condition variance retained by scGPT. Panel B evaluates state-specific radiation-response deltas using independently centred observed and predicted control profiles. Panel C tests whether scGPT preserves distances between AC-, MES-, NPC- and OPC-like centroids in a common 10-PC space. Panel D shows observed and predicted cells in one jointly fitted PCA basis.

In [ ]:
main_figure_paths = [
    OUTPUT_DIR / "figure_scgpt_split_composition.png",
    OUTPUT_DIR / "figure_scgpt_validation_trajectory.png",
    OUTPUT_DIR / "figure_scgpt_final_metrics.png",
    OUTPUT_DIR / "figure_scgpt_heterogeneity_recovery.png",
]

for figure_path in main_figure_paths:
    print(figure_path)
    display(Image(filename=str(figure_path)))

**Heterogeneity-recovery figure caption.** (A) Ratio of summed per-gene variance in predicted and observed expression across the 2,465 gene features, calculated separately for nominal test cells at each irradiated timepoint. (B) Pearson correlation between observed and predicted state-specific treatment-control delta profiles for AC-, MES-, NPC- and OPC-like cells; observed and predicted deltas use their respective state-matched control centroids. (C) Pairwise distances between malignant-state centroids in the first 10 components of a PCA fitted jointly to observed and predicted irradiated cells; the dashed line indicates equal separation. (D1-D2) Observed and predicted density contours overlaid in the same jointly fitted PCA coordinate system for each irradiated timepoint. Axes and aspect ratios are identical across both panels; circles and diamonds mark observed and predicted state centroids, respectively, and connecting lines show centroid displacement. The compact predicted densities, together with 1.58% and 2.51% variance retention and approximately 6% centroid-distance retention, show that strong pseudobulk agreement coexists with severe loss of single-cell heterogeneity.

## Biological Response Recovery

The top-gene bar chart compares observed and predicted mean expression changes for the strongest response genes. The pathway/programme figure then tests coordinated RT-response signatures and AC-, MES-, NPC- and OPC-like programmes. The RT-response panels use compact directional marker sets for p53/DNA damage, cell-cycle arrest, apoptosis, interferon/inflammation, oxidative stress, hypoxia and senescence/SASP. The GBM state panels use representative Neftel-informed markers. Exact genes retained in the 2,465-gene model feature space are exported in `pathway_gene_sets.csv`. Scores are averages of control-standardised expression values, and changes are calculated within each batch relative to its control sample. This is targeted signature scoring, not pathway enrichment.

In [ ]:
biological_figure_paths = [
    OUTPUT_DIR / "figure_scgpt_top20_deg_barcharts.png",
    OUTPUT_DIR / "figure_scgpt_pathway_score_recovery.png",
]

for figure_path in biological_figure_paths:
    print(figure_path)
    display(Image(filename=str(figure_path)))

**Pathway/programme figure caption.** Observed and predicted changes in targeted RT-response and GBM-state signature scores at 24 hours and one week after irradiation. Scores were computed as the mean control-standardised expression of the retained signature genes. For each of four batches, the irradiated sample score was referenced to the batch-matched control; small points show these batch-level changes, large markers show their means, horizontal error bars show SEM, and grey lines join observed and predicted means. Gene counts in the labels report the exact features available to each compact marker panel. This is a targeted signature-recovery analysis rather than a gene-set enrichment test.

## Appendix: Supporting Gene-Level Diagnostics

The top-20 boxplots are retained here because their boxes show observed single-cell distributions while each red marker is only a predicted mean. They assess average placement for selected genes, not recovery of predicted single-cell heterogeneity. In the linearity diagnostic, the residual-panel black line is a descriptive median calculated within equal-count bins of observed change; it is not a fitted quadratic trend. The quadratic statistic shown in the upper panels is only the gain in $r^2$ obtained by adding a squared term to the observed-versus-predicted relationship. The remaining plots provide complementary distributional and gene-level agreement checks.

In [ ]:
appendix_figure_paths = [
    OUTPUT_DIR / "figure_scgpt_observed_pseudobulk_expression_distributions.png",
    OUTPUT_DIR / "figure_scgpt_pseudobulk_expression_distributions.png",
    OUTPUT_DIR / "figure_scgpt_pseudobulk_expression_ecdf_qq.png",
    OUTPUT_DIR / "figure_scgpt_pseudobulk_expression_rank_bias.png",
    OUTPUT_DIR / "figure_scgpt_delta_scatter.png",
    OUTPUT_DIR / "figure_scgpt_delta_linearity_diagnostics.png",
    OUTPUT_DIR / "figure_scgpt_observed_delta_distributions.png",
    OUTPUT_DIR / "figure_scgpt_delta_distributions.png",
    OUTPUT_DIR / "figure_scgpt_top20_deg_boxplots.png",
]

for figure_path in appendix_figure_paths:
    print(figure_path)
    display(Image(filename=str(figure_path)))

## Draft Result Text

The exported metadata defined an approximately 80/10/10 nominal cell-level split, comprising 15,229 training cells, 1,904 validation cells and 1,904 test cells, with condition proportions preserved across the three files. An integrity audit nevertheless showed that both saved validation and test prediction arrays contained all 19,037 cells. Exact matching of truth-expression rows recovered a unique train, validation or test metadata assignment for every result row. Because the GEARS loaders were constructed from condition membership while every condition occurred in every nominal split, this run should be interpreted as a descriptive reconstruction analysis rather than a clean estimate of held-out generalisation.

Across the full saved evaluation arrays, scGPT showed high condition-level agreement, with an average Pearson correlation of 0.9911 and an average Pearson delta correlation of 0.9439. These pseudobulk summaries concealed substantial loss of single-cell structure. After restricting the exactly matched rows to the nominal test subset, predicted expression retained only 1.58% of observed within-condition variance at 24 hours and 2.51% at one week. Pairwise distances between AC-, MES-, NPC- and OPC-like centroids retained a mean of 5.9% and 6.1% of their observed separation, respectively, and predicted cells formed compact clusters in the jointly fitted PCA space.

Recovery of mean radiation-response profiles varied by malignant state. State-specific gene-level delta correlations ranged from 0.372 for AC-like cells to 0.868 for NPC-like cells at 24 hours, and from 0.469 for OPC-like cells to 0.782 for NPC-like cells at one week. Thus, the model recovered the average NPC-like response relatively well while failing to reproduce the broader cell-to-cell and between-state heterogeneity of the observed data.

Targeted pathway and state-programme score changes were more concordant at 24 hours than at one week. Across 11 compact signatures, observed and predicted score changes correlated at 0.910 at 24 hours and 0.840 at one week, with mean absolute errors of 0.042 and 0.060 SD units. Interferon/inflammatory and oxidative-stress responses were recovered in the correct positive direction at both timepoints. The one-week DNA-damage/p53 and OPC-like programme changes showed direction mismatches, highlighting specific biological effects that were not captured despite the high global pseudobulk correlations.

The top-20 DEG bar chart is retained as a gene-level mean-response summary. The corresponding boxplots are presented in the Appendix because their boxes show observed single-cell distributions while each red marker represents only one predicted mean; they therefore do not test heterogeneity recovery.